In [3]:
# Install Transformer library
!pip install transformers torch

In [4]:
#importing mmodel using transformer
from transformers import pipeline

In [5]:
chatbot = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.1",
    pad_token_id=2,
    max_new_tokens=300
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [6]:
#prompt engineering to make the responses friendly and clear
SYSTEM_PROMPT = """
You are a helpful medical assistant chatbot. Act like a top 1% helpful medical assistants. Give simple and friendly health information.

Safety Rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not give exact dosage or overdose limits.
- For medicine questions, do not simply say "yes". Say it may be used only as directed by a doctor/pharmacist or medicine label.
- For children, pregnancy, allergies, chronic illness, or serious symptoms, recommend consulting a doctor or pharmacist.
"""

danger_keywords = [
    "chest pain",
    "difficulty breathing",
    "suicide",
    "overdose",
    "severe bleeding"
]

In [7]:
#create safety filter function
def safety_filter(query):
    query = query.lower()
    for word in danger_keywords:
        if word in query:
            return True
    return False

In [8]:
def health_chatbot(user_query):

    if safety_filter(user_query):
        return "This may be serious. Please contact a doctor or emergency service immediately."

    prompt = f"""
    {SYSTEM_PROMPT}

    User: {user_query}

    Assistant:
    """

    response = chatbot(
        prompt,
        temperature=0.5,
        do_sample=True,
        return_full_text=False,
        eos_token_id=2
    )

    answer = response[0]["generated_text"].strip()

    # Remove extra generated chat
    stop_words = ["User:", "Assistant:"]

    for word in stop_words:
        if word in answer:
            answer = answer.split(word)[0]

    return answer.strip()

In [9]:
# Create chatbot function
def health_chatbot(user_query):
  if safety_filter(user_query):
        return "This may be a serious medical emergency. Please contact a doctor or emergency service immediately."

  prompt = SYSTEM_PROMPT + "\nUser: " + user_query + "\nAssistant:"

  response = chatbot(
        prompt,
        do_sample=True,
        temperature=0.7,
        return_full_text=False
    )

  answer = response[0]["generated_text"].strip()

  return answer


In [10]:
# Test Questions
questions = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?"
]

for q in questions:

    print("\nUser:", q)
    print("Bot:", health_chatbot(q))
    print("-" * 50)


User: What causes a sore throat?
Bot: A sore throat can be caused by many things. It's often a symptom of a viral infection like the common cold. It can also be due to a bacterial infection, tonsillitis, mononucleosis, strep throat, or other conditions. If your symptoms are severe or persist, it's important to consult a doctor.
--------------------------------------------------

User: Is paracetamol safe for children?
Bot: It may be used in children as directed by a doctor or pharmacist. However, always consult a doctor or pharmacist before giving any medications to a child.
--------------------------------------------------


In [11]:
#to take real time input and test model
print("Health Assistant Chatbot")
print("Type your health question below. Type 'quit' to exit.")

user_query = ""

while user_query != "q":

    user_query = input("\n Hey Wassup please let us know how i can assist you: ")

    if user_query == "q":
        print("Thanks for choosing General Health Query Chatbot")
        break

    response = health_chatbot(user_query)
    print("\n \n Bot:", response)


Health Assistant Chatbot
Type your health question below. Type 'quit' to exit.

 Hey Wassup please let us know how i can assist you: What causes a sore throat?

 
 Bot: A sore throat can be caused by various reasons such as cold and flu, bacterial or viral infection, postnasal drip, or irritation from smoking, alcohol, or acidic foods. It's important to consult a doctor if symptoms persist or worsen, or if you experience fever, severe pain, or difficulty swallowing.

 Hey Wassup please let us know how i can assist you: What causes a sore throat?

 
 Bot: A sore throat can be caused by various factors such as viral infections like the common cold or flu, bacterial infections, tonsillitis, strep throat, and allergies. It's always best to consult a doctor or pharmacist for proper diagnosis and treatment.

 Hey Wassup please let us know how i can assist you: What are common symptoms of dehydration?

 
 Bot: Common symptoms of dehydration include thirst, dry mouth, dark urine, fatigue, head